## Requirements

Place these files in the SAME directory:
```bash
main.py
build_vectors_from_main_py.ipynb
```

Export HF token before launching:
```bash
export HF_TOKEN="your_token"
```


In [ ]:
# Cell 1 — Install dependencies

!pip install -q transformers==4.40.0 torch scikit-learn accelerate huggingface_hub

print("✓ Dependencies installed")


In [ ]:
# Cell 2 — Authenticate with Hugging Face

import os
from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN not found. Run:\n"
        "export HF_TOKEN='your_token'"
    )

login(token=HF_TOKEN)

print("✓ Hugging Face login successful")


In [ ]:
# Cell 3 — Verify GPU + versions
import torch, transformers
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM         : {vram_gb:.1f} GB')
    if vram_gb < 13.5:
        print('WARNING: Llama-2-7b-hf needs ~14 GB VRAM. Consider upgrading to A100.')
    else:
        print('✓ Sufficient VRAM for Llama-2-7b-hf in bfloat16')
else:
    print('WARNING: No GPU — this will be extremely slow. Runtime → Change runtime type → A100 GPU')

In [ ]:
# Cell 4 — Verify main.py exists

from pathlib import Path

main_file = Path("main.py")

if not main_file.exists():
    raise FileNotFoundError(
        "main.py not found in current directory."
    )

print(f"✓ Found main.py at: {main_file.resolve()}")


In [ ]:
# Cell 5 — Configure environment and import from main.py

import os
import importlib.util
from pathlib import Path

# Environment variables
os.environ["LOCAL_MODEL_NAME"] = "meta-llama/Llama-2-7b-hf"
os.environ["STEER_LAYER"] = "20"
os.environ["STYLE_CACHE_DIR"] = ".style_cache"

# Create cache directory
Path(".style_cache").mkdir(exist_ok=True)

# Dynamically import main.py
spec = importlib.util.spec_from_file_location("main_module", "main.py")
main_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(main_module)

print("✓ main.py imported successfully")

print("=" * 60)
print("Model      :", os.environ["LOCAL_MODEL_NAME"])
print("Steer layer:", os.environ["STEER_LAYER"])
print("Cache dir  :", os.environ["STYLE_CACHE_DIR"])
print("=" * 60)


In [ ]:
# Cell 6 — Build EMPATHETIC vector using main.py

print("Building EMPATHETIC vector...")

vec_emp = main_module.build_style_vector(
    style="empathetic",
    method="pca"
)

print("\n✓ EMPATHETIC vector built successfully")


In [ ]:
# Cell 7 — Build FORMAL vector using main.py

print("Building FORMAL vector...")

vec_formal = main_module.build_style_vector(
    style="formal",
    method="pca"
)

print("\n✓ FORMAL vector built successfully")


In [ ]:
# Cell 8 — Verify vectors are valid
import torch

sim = torch.nn.functional.cosine_similarity(
    vec_emp.unsqueeze(0), vec_form.unsqueeze(0)
).item()

print('='*55)
print(f'  empathetic : norm={vec_emp.norm():.6f}  shape={list(vec_emp.shape)}')
print(f'  formal     : norm={vec_form.norm():.6f}  shape={list(vec_form.shape)}')
print(f'  cosine sim : {sim:.4f}')
print()

assert abs(vec_emp.norm().item()  - 1.0) < 0.01, 'empathetic vector not unit-norm!'
assert abs(vec_form.norm().item() - 1.0) < 0.01, 'formal vector not unit-norm!'
assert vec_emp.shape[0]  == 4096, f'Expected hidden_dim=4096 for Llama-2-7b, got {vec_emp.shape[0]}'
assert vec_form.shape[0] == 4096, f'Expected hidden_dim=4096 for Llama-2-7b, got {vec_form.shape[0]}'
assert sim < 0.5, f'Cosine similarity {sim:.3f} too high — vectors may not be distinct!'

if sim < 0:
    print('✓ Cosine similarity is NEGATIVE — vectors point in opposite directions (expected)')
else:
    print(f'  Cosine similarity {sim:.3f} > 0 — vectors diverge but are not opposite.')
    print('  This is acceptable — the important thing is they are distinct directions.')

import os
cache_files = list(os.listdir('.style_cache'))
print(f'\nCached files in .style_cache/:')
for f in cache_files:
    size = os.path.getsize(f'.style_cache/{f}') / 1024
    print(f'  {f}  ({size:.1f} KB)')
print('\n✓ All checks passed!')
print('  Note: Each pkl is ~16 KB (4096 float32 values) — much larger than Qwen\'s 3.5 KB (896 floats)')

In [ ]:
# Cell 9 — Quick smoke test: does the hook actually steer generation?
import torch
from style_vectors import _get_layer

ALPHA_TEST = 15.0   # lower than Qwen's 20.0 — 7B model is more sensitive
test_prompt = (
    '[INST] You are a customer support agent. '
    'Write a support reply to this complaint: '
    'My <PRODUCT> order <ORDER_ID> has an <ISSUE>. [/INST] '
)

def generate_steered(prompt, style_vec, alpha, label):
    device = next(model.parameters()).device
    # Upcast to float32 for hook arithmetic
    vec    = style_vec.to(device=device, dtype=torch.float32)
    layer  = _get_layer(model)

    def _hook(module, inp, output):
        h = output[0] if isinstance(output, tuple) else output
        h_f32 = h.float()
        h_f32[:, -1, :] = h_f32[:, -1, :] + alpha * vec
        steered = h_f32.to(h.dtype)  # cast back to bfloat16
        return (steered,) + output[1:] if isinstance(output, tuple) else steered

    handle = layer.register_forward_hook(_hook)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    try:
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
            )
    finally:
        handle.remove()

    new_ids = out[0][inputs.input_ids.shape[1]:]
    text    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    print(f'\n[{label}]  {text[:300]}')
    return text

print('=== Smoke Test: Same prompt, two style vectors ===')
print(f'Prompt: {test_prompt}')

r_emp  = generate_steered(test_prompt, vec_emp,  ALPHA_TEST, 'EMPATHETIC')
r_form = generate_steered(test_prompt, vec_form, ALPHA_TEST, 'FORMAL')

print('\n=== Smoke test complete — check that the two responses have different tones ===')

In [ ]:
# Cell 10 — Verify saved vectors

from pathlib import Path

cache_dir = Path(".style_cache")

print("=" * 60)
print("Stored vectors")
print("=" * 60)

for file in cache_dir.glob("*"):
    size_kb = file.stat().st_size / 1024
    print(f"{file.name:<45} {size_kb:.2f} KB")

print("=" * 60)
print("Absolute cache path:")
print(cache_dir.resolve())
